# 08 Confirmation of the frontier configs: all usable years x several block-split folds
Run after 07. For each region it takes ~8 configs spread over the cost range (the Pareto frontier plus fill-ins) and re-scores them on
every usable year (not just the search years) and on `N_FOLDS` different block assignments (fold 0 = the split used in 04-07).
Output (`{REGION}_confirmation.csv`, `{REGION}_tier_summary.csv`) is the basis of the group guidelines: expected F1, year-to-year range, spatial-split noise.
Classes with fewer than `ss.MIN_POLYS` polygons keep the same pinned polygons in every fold, so their spread is NOT covered; they are reported as low-confidence.

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import spatial_split as ss
from year_check import years_available, pick_configs, evaluate, report, tier_summary

REGIONS = os.environ.get('REGIONS', 'HIM,PEN,DES,IGP,WGR,ICR,NEI').split(',')
N_FOLDS = int(os.environ.get('N_FOLDS', 4))        # folds 0..N_FOLDS-1 (fold 0 = the 04-07 split)
N_CONFIGS = int(os.environ.get('N_CONFIGS', 8))
MIN_GATED_ROWS = 20_000                            # 'enough data' rule for a year to count
N_JOBS = int(os.environ.get('RF_JOBS', 4))
N_YEARS = int(os.environ.get('N_YEARS', 14))      # evenly spaced subset of the usable years (0 = all); the search years are always kept
N_SEEDS = int(os.environ.get('N_SEEDS', 1))       # training draws per config x fold x year
artifact_dir = Path('data/feature_selection_artifacts'); out_dir = Path('data/optuna_regional_gee_configs')

old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
ref_df = pd.read_csv('data/classwise_samples_reference_ratios.csv'); ref_df['new_code'] = new_codes
ref_cols = [c for c in ref_df.columns if c not in ['l2_code', 'l2_name', 'l1_parent', 'old_code', 'new_code']]
parent_ref_df = ref_df.groupby('new_code')[ref_cols].sum()

In [ ]:
# Cell 3: build the extra folds (fold 0 already exists from notebook 03), then evaluate
for region in REGIONS:
    print(f"\n{'='*60}\n{region}\n{'='*60}")
    for f in range(1, N_FOLDS):
        ss.build_block_splits(region, fold=f)
    all_ranked = json.load(open(artifact_dir / f'{region}_ranked_features.json'))['ranked_features']
    uncorr_ranked = json.load(open(artifact_dir / f'{region}_uncorrelated_features.json'))['uncorrelated_features']
    ref = parent_ref_df[region] if region in parent_ref_df.columns else None
    cnt = years_available(region); years = [int(y) for y in cnt.year[cnt.gated_rows >= MIN_GATED_ROWS] if int(y) in ss.study_years(region)]
    if N_YEARS and len(years) > N_YEARS:
        pick = {years[int(round(i))] for i in np.linspace(0, len(years) - 1, N_YEARS)}
        years = sorted(pick | {y for y in ss.search_years(region) if y in years})
    cfgs = pick_configs(out_dir / f'{region}_all_trials.json', n=N_CONFIGS)
    print(f'{len(cfgs)} configs, {len(years)} years, {N_FOLDS} folds; scored classes {ss.scored_classes(region)[0]}')
    res = evaluate(region, cfgs, years, all_ranked, uncorr_ranked, ref, n_jobs=N_JOBS, seeds=tuple(range(N_SEEDS)), folds=tuple(range(N_FOLDS)))
    res.to_csv(out_dir / f'{region}_confirmation.csv', index=False)
    tiers = tier_summary(res, cfgs)
    tiers.to_csv(out_dir / f'{region}_tier_summary.csv', index=False)
    print(tiers.round(4).to_string(index=False))
    report(res, search_years=ss.search_years(region))